# Port binding

**Objective:** Export the web service through a host and port supplied by the runtime environment.

## Simple version

The application does not install or configure an external web server; it starts its own HTTP server on the assigned port.

In [ ]:
host = "0.0.0.0"
port = 8000

command = f"uvicorn app:app --host {host} --port {port}"
print(command)

## Polished version

Configuration owns the binding while an app factory owns the HTTP application. The runtime can change the port without changing code.

In [ ]:
from collections.abc import Mapping
from dataclasses import dataclass

import httpx
from fastapi import FastAPI


@dataclass(frozen=True)
class Binding:
    host: str
    port: int

    @classmethod
    def from_env(cls, env: Mapping[str, str]) -> "Binding":
        port = int(env.get("PORT", "8000"))
        if not 1 <= port <= 65_535:
            raise ValueError("PORT must be between 1 and 65535")
        return cls(host=env.get("HOST", "0.0.0.0"), port=port)

    def command(self) -> list[str]:
        return [
            "uvicorn",
            "app:app",
            "--host",
            self.host,
            "--port",
            str(self.port),
        ]


def create_app() -> FastAPI:
    app = FastAPI()

    @app.get("/health")
    async def health() -> dict:
        return {"status": "ok"}

    return app


binding = Binding.from_env({"PORT": "8080"})
app = create_app()
transport = httpx.ASGITransport(app=app)

async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    response = await client.get("/health")

print(response.json())
print("Run:", " ".join(binding.command()))

## Applied in this repository

Both project Makefiles start a self-contained FastAPI application with Uvicorn. Production supplies the host and port through the process environment.